<a href="https://colab.research.google.com/github/anitabudhiraja/DeepLearning/blob/main/practical_5_imbalanced_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Techniques (DOAI250006)
## Institution: NIELIT ROPAR
## Practical 5: Handling and evaluating imbalance data using resampling, ensemble method and custom loss function

This notebook demonstrates how to address class imbalance—a common problem where one class heavily outnumbers another—using three techniques: resampling (SMOTE), class-weighted loss functions, and an ensemble approach.

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


### 1. Generating an Imbalanced Dataset
We create a synthetic dataset where 95% of the samples belong to Class 0 and only 5% belong to Class 1.

In [ ]:
# Create an imbalanced dataset (95% majority, 5% minority)
X, y = make_classification(n_samples=10000, n_features=20, n_classes=2,
                           weights=[0.95, 0.05], random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Set Class Distribution:")
print(f"Class 0: {np.sum(y_train == 0)}, Class 1: {np.sum(y_train == 1)}")

Training Set Class Distribution:
Class 0: 7576, Class 1: 424


### 2. Method 1: Resampling using SMOTE (Synthetic Minority Over-sampling Technique)
SMOTE generates synthetic examples of the minority class so that the training data becomes balanced.

In [ ]:
# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Resampled Training Set Class Distribution:")
print(f"Class 0: {np.sum(y_train_resampled == 0)}, Class 1: {np.sum(y_train_resampled == 1)}")

# Build and train a simple Neural Network on the resampled data
model_smote = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model_smote.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_smote.fit(X_train_resampled, y_train_resampled, epochs=5, verbose=0)

# Evaluate
y_pred_smote = (model_smote.predict(X_test) > 0.5).astype(int)
print("\nClassification Report (SMOTE):")
print(classification_report(y_test, y_pred_smote))

Resampled Training Set Class Distribution:
Class 0: 7576, Class 1: 7576


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Classification Report (SMOTE):
              precision    recall  f1-score   support

           0       0.99      0.92      0.96      1884
           1       0.42      0.92      0.58       116

    accuracy                           0.92      2000
   macro avg       0.71      0.92      0.77      2000
weighted avg       0.96      0.92      0.93      2000



### 3. Method 2: Handling Imbalance using Custom Loss Weights
Instead of changing the data, we tell the algorithm to penalize mistakes on the minority class more heavily by passing `class_weight` to the model fit function.

In [ ]:
# Calculate class weights: Higher weight for the minority class (1)
total = len(y_train)
pos = np.sum(y_train == 1)
neg = total - pos

weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}
print(f"Calculated Class Weights: {class_weight}")

# Build and train the model using class weights
model_weighted = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model_weighted.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_weighted.fit(X_train, y_train, epochs=5, class_weight=class_weight, verbose=0)

# Evaluate
y_pred_weighted = (model_weighted.predict(X_test) > 0.5).astype(int)
print("\nClassification Report (Class Weights):")
print(classification_report(y_test, y_pred_weighted))

Calculated Class Weights: {0: np.float64(0.5279831045406547), 1: np.float64(9.433962264150942)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Classification Report (Class Weights):
              precision    recall  f1-score   support

           0       0.99      0.88      0.93      1884
           1       0.32      0.91      0.48       116

    accuracy                           0.88      2000
   macro avg       0.66      0.89      0.70      2000
weighted avg       0.95      0.88      0.91      2000



### 4. Method 3: Ensemble Method (Balanced Random Forest)
Using an ensemble learning method specifically designed for imbalanced datasets. It randomly under-samples each bootstrap sample to balance the trees.

In [ ]:
# Train a Balanced Random Forest Classifier
brf_model = BalancedRandomForestClassifier(n_estimators=100, random_state=42)
brf_model.fit(X_train, y_train)

# Evaluate
y_pred_brf = brf_model.predict(X_test)
print("\nClassification Report (Balanced Random Forest Ensemble):")
print(classification_report(y_test, y_pred_brf))


Classification Report (Balanced Random Forest Ensemble):
              precision    recall  f1-score   support

           0       1.00      0.93      0.96      1884
           1       0.44      0.95      0.60       116

    accuracy                           0.93      2000
   macro avg       0.72      0.94      0.78      2000
weighted avg       0.96      0.93      0.94      2000

